# Feature Import Report — `CartPole-v1-PolicyBased-pytorch` → DDORWA (TSP) Project

**Author:** Ali Momen  
**Source repository (imported from):** [`https://github.com/amomen9/CartPole-v1-PolicyBased-pytorch`](https://github.com/amomen9/CartPole-v1-PolicyBased-pytorch)  
**Target repository (this project):** [`https://github.com/amomen9/DDORWA_Project`](https://github.com/amomen9/DDORWA_Project)  
**Date:** 2026-05-30

---

## Background

This project and the `CartPole-v1-PolicyBased-pytorch` repository are two divergent forks of the same
reinforcement-learning code base. The CartPole fork matured a set of training-pipeline features that
this TSP fork had only partially adopted. This report documents the features that were **discovered in
the CartPole fork and imported into this project**, classified by concept, with a brief explanation of
each and the adaptations that the TSP problem's nature made mandatory.

> **Adaptation principle.** Features were deployed as faithfully as possible. Modifications were made
> only where the TSP project's nature forced them — chiefly because the TSP **action space depends on the
> instance size** (number of cities), and the project keeps a **per-repetition / per-setting checkpoint
> layout** with a dict payload that the existing evaluator (`Use_Trained_Model.py`) relies on.

Features the TSP fork had **already** imported before this work (not re-imported here): Excel disk-reuse
(`use_existing_disk_data`), fixed-length PPO rollouts, smoothing windows, evaluation intervals, parallel
repetitions, and a domain-specific DP-equivalency evaluator.

## 1. Checkpoint Persistence & Reuse

**Imported from:** CartPole `Checkpointing.py`.  
**Now lives in:** `Library/Library_checkpointing.py`.

A self-contained checkpoint module that adds, on top of the raw `.pt` files the project already saved:

- **JSON metadata sidecars** — every checkpoint `*.pt` gets a sibling `*.txt` recording the
  hyperparameters, `n_actions`, and the cumulative `n_timesteps` (atomic, normalized writes).
- **Exact-match resolution** — a saved network is reused only if its sidecar matches the requested
  configuration exactly (provenance-only keys excluded).
- **Loose-match resolution** (`skip_selection_hyperparameter_match`) — ignore the exact hyperparameter
  match and instead pick the compatible candidate with the **largest accumulated `n_timesteps`** (the
  natural choice for resuming the longest-trained snapshot).
- **Atomic, non-overwriting saves** — a new run never destroys an older checkpoint; collisions become
  `*_1.pt`, `*_2.pt`, …

**Mandatory TSP adaptation:** the *strict-match gate* keys on **`n_actions`** (which encodes the number
of cities) instead of CartPole's training-truncation field, because an actor trained for a different TSP
size is weight-incompatible with the current environment. Filenames keep the project's existing
`actor_rep{N}[_S{id}]` / `critic_rep{N}[_S{id}]` stems rather than CartPole's architecture-signature
names, and the payload keeps the project's `{"state_dict", "n_actions", "actor_hidden_nn", …}` dict shape
so the legacy evaluator still loads it unchanged.

## 2. Training Continuation

**Imported from:** CartPole `Library.py` (worker load/continue/save logic) + `Experiment.py` flags.  
**Now lives in:** `Library/Library_experiment_orchestrator.py` (per-repetition worker `_run_single_repetition` and the
dispatch), `Library/Helper_parallel_run.py` (flag threading), `Experiment.py` (`global_config` flags).

When `use_saved_disk_networks_checkpoints = True`, each training repetition **loads a matching saved
actor (and critic) and continues training from it**, then writes the network back to the *same* file with
the timestep counter accumulated (e.g. `200000 + 250000 = 450000`). When the flag is off, behaviour is
the project's original in-place save — now also emitting the metadata sidecar.

An **up-front load report** is printed before the worker pool starts, listing for each (setting,
repetition) whether a matching checkpoint will be continued or training will start fresh — emitted before
the child-process progress bars so the lines are not interleaved.

**Mandatory TSP adaptation:** CartPole saves only at repetition 0; the TSP pipeline trains and persists
**every repetition independently** (each `actor_rep{N}_S{id}` file continues its own lineage), and the
critic is persisted/continued only when the agent actually exposes an `nn.Module` critic.

## 3. Checkpoint Evaluation

**Imported from:** CartPole `Evaluate Checkpoints.py` + `run_actor_checkpoint_evaluation_exhaustive`.  
**Now lives in:** `Evaluate Checkpoints.py` (project root) + `Library/Library_checkpoint_eval.py`.

A standalone entry point that **evaluates every saved checkpoint of each enabled algorithm exhaustively**
— each checkpoint is rolled out for `n_episodes` episodes under one or more action-selection methods, and
per-checkpoint statistics are reported (with an optional bar chart of mean performance and a DP-optimal
reference line).

**Mandatory TSP adaptation:** the routine reuses the project's own TSP rollout machinery
(`Use_Trained_Model._run_loop_on_matrix`, `Policy_NN`) and reports **tour cost** (lower is better)
against a chosen reference matrix — `expected` / `deterministic` / `stochastic` — instead of CartPole's
fixed-environment episode returns. Action-selection methods are the TSP analogues `sample` / `argmax`
(vs CartPole's `softmax` / `argmax`), and the entry point first builds a TSP environment from a saved
matrix trio exactly as `Experiment.py` does.

## 4. Continuation Analysis

**Imported from:** CartPole `Helper.build_returns_summary_table` + the "Trial Continuation Analysis"
output routing in `Library.py`.  
**Now lives in:** `Library/Library_continuation_analysis.py` + routing in `Library/Library_experiment_orchestrator.py`
and an `out_dir` argument on `Library/Library_plotting.LearningCurvePlot.save`.

After a run, a **returns-summary table** is produced — per (algorithm, setting) it pools every repetition
and every evaluation point into an overall mean/std and computes a **last-10% mean/std** (the converged
performance estimate). In checkpoint-reuse / continuation mode the table is also written to disk
(`results_summary.csv` / `.md`) and the learning-curve figures are routed to a separate
**`Trial Continuation Analysis/`** directory so continued-run diagnostics do not mix with from-scratch
results.

**Mandatory TSP adaptation:** the table is computed directly from the TSP orchestrator's own structures
(`setting_results` tuples and job `curve_label`s) as a self-contained module, rather than depending on the
CartPole fork's legend/exclusion helper subtree.

## 5. Experiment Orchestration / Ablation Scripts

**Imported from:** CartPole's family of dedicated `Experiment_*.py` sweep scripts.  
**Now lives in:** `Experiment_Gamma.py`, `Experiment_ActorLR.py`, `Experiment_CriticLR.py`,
`Experiment_ActorNN.py`, `Experiment_CriticNN.py`, `Experiment_Epochs.py`, `Experiment_GAE.py`,
`Experiment_Clip.py`, `Experiment_Rollout.py`, `Experiment_EvalMore.py`, plus an `overrides` hook on
`Experiment.Test_TSP`.

Each ablation / hyperparameter sweep can be reproduced with a **single command** (e.g.
`python Experiment_Gamma.py` sweeps the PPO discount factor).

**Mandatory TSP adaptation:** the TSP project keeps its configuration *inside* `Test_TSP()` (unlike the
CartPole fork's module-level config), so instead of duplicating the whole pipeline in every script, a
lightweight `overrides=` argument was added to `Test_TSP`. Each `Experiment_*.py` is then a thin wrapper
that shallow-merges one swept list into the relevant config section, keeping the "one command per
ablation" behaviour without copy-pasting the orchestration code.

## Summary of files added / modified

| File | Status | Feature group |
| --- | --- | --- |
| `Library/Library_checkpointing.py` | **added** | 1. Persistence & reuse |
| `Library/Library_experiment_orchestrator.py` | modified | 1, 2, 4 (flags, worker load/continue/save, report, summary, plot routing) |
| `Library/Helper_parallel_run.py` | modified | 2 (thread reuse flags to the worker) |
| `Experiment.py` | modified | 2, 5 (global flags + `overrides` hook) |
| `Library/Library_checkpoint_eval.py` | **added** | 3. Checkpoint evaluation |
| `Evaluate Checkpoints.py` | **added** | 3. Checkpoint evaluation (entry point) |
| `Library/Library_continuation_analysis.py` | **added** | 4. Continuation analysis |
| `Library/Library_plotting.py` | modified | 4 (`out_dir` for plot routing) |
| `Experiment_*.py` (10 scripts) | **added** | 5. Ablation scripts |

## How to use

```bash
# Train with checkpoint continuation (resume + accumulate timesteps):
#   set use_saved_disk_networks_checkpoints = True in Experiment.py global_config
python Experiment.py

# Evaluate every saved checkpoint exhaustively:
python "Evaluate Checkpoints.py"

# Reproduce a single PPO ablation in one command:
python Experiment_Gamma.py        # discount factor sweep
python Experiment_ActorLR.py      # actor learning-rate sweep
python Experiment_Clip.py         # PPO clip-epsilon sweep
# ... etc.
```